In [ ]:
import os
from google.colab import drive
from tqdm.auto import tqdm

unzipped_images_dir = '/dev/shm/extract_filtered'
zip_path = '/content/drive/MyDrive/Colab Notebooks/extract_filtered.zip'
expected_image_count = 30200

def count_files_with_progress(directory, description):
    file_count = 0
    for root, dirs, files in tqdm(os.walk(directory), desc=description, unit="dir"):
        file_count += len(files)
    return file_count

# Check if images are already extracted
if os.path.exists(unzipped_images_dir) and os.listdir(unzipped_images_dir):
    current_image_count = count_files_with_progress(unzipped_images_dir, "Counting existing images")
    if current_image_count == expected_image_count:
        print(f"✅ Images already extracted to {unzipped_images_dir}. Skipping extraction. ({current_image_count} files found)")
    else:
        print(f"⚠️ Images directory {unzipped_images_dir} exists but contains {current_image_count} files instead of the expected {expected_image_count}.")
        user_response = input("Do you want to re-extract the images? (yes/no): ").strip().lower()
        if user_response == 'yes':
            print("Proceeding with re-extraction...")
            drive.mount('/content/drive')
            if os.path.exists(zip_path):
                print("✅ Zip file found! Starting re-extraction...")
                os.makedirs('/dev/shm', exist_ok=True)
                !unzip -o -q "{zip_path}" -d /dev/shm
                final_image_count = count_files_with_progress(unzipped_images_dir, "Counting re-extracted images")
                if final_image_count == expected_image_count:
                    print(f"🎉 Re-extraction complete and verified! ({final_image_count} files found)")
                else:
                    print(f"❌ Re-extraction complete, but found {final_image_count} files instead of {expected_image_count}.")
            else:
                print(f"❌ Error: Zip file not found at {zip_path}")
                print("Double check if there is a typo in 'Colab Notebooks' or the filename.")
        else:
            print(f"Skipping re-extraction. Using existing images ({current_image_count} files).")
else:
    print(f"Images not found in {unzipped_images_dir}. Proceeding with extraction...")
    drive.mount('/content/drive')

    if os.path.exists(zip_path):
        print("✅ Zip file found! Starting extraction into /dev/shm (RAM)...")
        os.makedirs('/dev/shm', exist_ok=True)
        !unzip -q "{zip_path}" -d /dev/shm

        final_image_count = count_files_with_progress(unzipped_images_dir, "Counting extracted images")
        if final_image_count == expected_image_count:
            print(f"🎉 Extraction complete and verified! ({final_image_count} files found)")
        else:
            print(f"❌ Extraction complete, but found {final_image_count} files instead of {expected_image_count}.")
    else:
        print(f"❌ Error: Zip file not found at {zip_path}")
        print("Double check if there is a typo in 'Colab Notebooks' or the filename.")

In [ ]:
!ls

In [ ]:
!cp /content/drive/MyDrive/Colab\ Notebooks/extract_filtered.zip /content/

In [ ]:
!find /dev/shm/extract_filtered -type f | wc -l

In [ ]:
from pathlib import Path

IMAGES_PATH = Path("/dev/shm/extract_filtered")

def index_labeled_images(images_path=IMAGES_PATH, max_per_class=3000):
    images_path = Path(images_path)
    labeled_images = {}
    if not images_path.exists():
        return labeled_images

    for cloud_dir in sorted(p for p in images_path.iterdir() if p.is_dir()):
        count = 0
        for img_path in sorted(p for p in cloud_dir.iterdir() if p.is_file()):
            if count >= max_per_class:
                break
            labeled_images[img_path.name] = {
                "label": cloud_dir.name,
                "path": str(img_path)
            }
            count += 1

    return labeled_images

In [ ]:
labeled_images = index_labeled_images()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
from tqdm.auto import tqdm

def extract_labels(labeled_images):
    labels = []
    paths = []
    for (i, image_name) in enumerate(labeled_images):
        path = labeled_images[image_name]['path']
        paths.append(path)
        label = labeled_images[image_name]['label']
        labels.append(label)

    return paths, np.array(labels)

In [ ]:
paths, labels = extract_labels(labeled_images)

In [ ]:
print(labels)

In [ ]:
print(paths)

In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

In [ ]:
# augmentation

import torch
import torchvision.transforms.v2 as T
from torchvision.transforms.v2 import InterpolationMode

IMG_SIZE = (224, 224)  # required by ViT_B_16 SWAG weights
MAX_FRAC = 0.14

border_translation = T.RandomAffine(
    degrees=0,
    translate=(MAX_FRAC, MAX_FRAC),
    interpolation=InterpolationMode.BILINEAR,
    fill=0
)

class WrapTranslation:
    """Circular shift by a random fraction of the image size — picklable for num_workers > 0."""
    def __init__(self, max_frac):
        self.max_frac = max_frac

    def __call__(self, x):
        h, w = x.shape[-2], x.shape[-1]
        shift_h = int(torch.randint(-int(self.max_frac * h), int(self.max_frac * h) + 1, (1,)).item())
        shift_w = int(torch.randint(-int(self.max_frac * w), int(self.max_frac * w) + 1, (1,)).item())
        return torch.roll(x, shifts=(shift_h, shift_w), dims=(-2, -1))

wrap_translation = WrapTranslation(MAX_FRAC)

stacked = T.Compose([
    T.RandomChoice([wrap_translation, border_translation]),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.15),
    T.RandomAffine(
        degrees=20, scale=(0.90, 1.10),
        interpolation=InterpolationMode.BILINEAR, fill=0
    ),
    T.RandomResizedCrop(
        size=IMG_SIZE, scale=(0.80, 1.00), ratio=(0.90, 1.10),
        interpolation=InterpolationMode.BILINEAR
    ),
])

one_of = T.RandomChoice([
    wrap_translation,
    border_translation,
    T.RandomChoice([T.RandomHorizontalFlip(p=1.0), T.RandomVerticalFlip(p=1.0)]),
    T.RandomRotation(degrees=20, interpolation=InterpolationMode.BILINEAR, fill=0),
])

normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transforms = T.Compose([
    T.Resize(IMG_SIZE, interpolation=InterpolationMode.BILINEAR),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.RandomChoice([stacked, one_of]),
    normalize,
])

eval_transforms = T.Compose([
    T.Resize(IMG_SIZE, interpolation=InterpolationMode.BILINEAR),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    normalize,
])

In [ ]:
from torch.utils.data import DataLoader
import torchvision
import torch.nn as nn
from functools import partial

In [ ]:
torchvision.models.list_models()

In [ ]:
list(torchvision.models.get_model_weights("vit_b_16"))

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

weights = torchvision.models.ViT_B_16_Weights.IMAGENET1K_SWAG_LINEAR_V1
model = torchvision.models.vit_b_16(weights=weights).to(device)

In [ ]:
import torchvision.transforms.v2 as T

transforms = weights.transforms()
# transforms = T.Compose([
#     T.RandomHorizontalFlip(p=0.5),
#     T.RandomRotation(degrees=30),
#     T.RandomResizedCrop(size=(384, 384), scale=(0.8, 1.0)),
#     T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
#     T.ToImage(),
#     T.ToDtype(torch.float32, scale=True),
#     T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
# ])

In [ ]:
print(transforms)

In [ ]:
from sklearn.preprocessing import LabelEncoder

def encode_labels(labels):
    ordinal_encoder = LabelEncoder()
    encoded_labels = ordinal_encoder.fit_transform(labels)
    class_names = ordinal_encoder.classes_
    return encoded_labels, class_names


In [ ]:
encoded_labels, class_names = encode_labels(labels)

In [ ]:
print(class_names)

In [ ]:
print(encoded_labels)

In [ ]:
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedShuffleSplit
from PIL import Image
import numpy as np

class MyImages(Dataset):
    def __init__(self, paths, encoded_labels, split="train", test_size=0.2, val_size=0.1, random_state=42, transform=None,
                 split_indices=None):

        if split not in {None, "train", "val", "test"}:
            raise ValueError(f"split must be one of {{'None','train','val','test'}}, got {split!r}")

        self.transform = transform

        paths = np.array(list(paths))
        encoded_labels = np.array(list(encoded_labels))

        n = len(paths)
        if n != len(encoded_labels):
            raise ValueError(f"paths and encoded_labels must have same length, got {n} and {len(labels)}")

        if split_indices is None:
            sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
            trainval_idx, test_idx = next(sss1.split(paths, encoded_labels))

            trainval_fraction = 1.0 - test_size
            val_within_trainval = val_size / trainval_fraction

            sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_within_trainval, random_state=random_state)

            trainval_paths = paths[trainval_idx]
            trainval_labels = encoded_labels[trainval_idx]

            train_rel_idx, val_rel_idx = next(sss2.split(trainval_paths, trainval_labels))
            train_idx = trainval_idx[train_rel_idx]
            val_idx = trainval_idx[val_rel_idx]

            split_indices = {"train": train_idx, "val": val_idx, "test": test_idx}

        idx = split_indices[split]

        self.paths = paths[idx].tolist()
        self.encoded_labels = encoded_labels[idx].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.encoded_labels[idx]


In [ ]:
from functools import partial

DefaultCloudImages = partial(
    MyImages,
    paths=paths,
    encoded_labels=encoded_labels,
    transform=transforms,
    test_size=0.2,
    val_size=0.1,
    random_state=42
)

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np

paths_arr = np.array(list(paths))
labels_arr = np.array(list(encoded_labels))

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
trainval_idx, test_idx = next(sss1.split(paths_arr, labels_arr))

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.1/0.8, random_state=42)
train_rel_idx, val_rel_idx = next(sss2.split(paths_arr[trainval_idx], labels_arr[trainval_idx]))

split_indices = {
    "train": trainval_idx[train_rel_idx],
    "val":   trainval_idx[val_rel_idx],
    "test":  test_idx,
}

train_set = MyImages(paths=paths, encoded_labels=encoded_labels,
                     split="train", transform=train_transforms,
                     split_indices=split_indices)

valid_set = MyImages(paths=paths, encoded_labels=encoded_labels,
                     split="val", transform=eval_transforms,
                     split_indices=split_indices)

test_set  = MyImages(paths=paths, encoded_labels=encoded_labels,
                     split="test", transform=eval_transforms,
                     split_indices=split_indices)

In [ ]:
train_set[0]

In [ ]:
len(train_set)
len(valid_set)
len(test_set)

In [ ]:
display_transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.CenterCrop(500),
])

# create a dataset instance for the train split (use MyImages to request a split)
clouds_to_display = DefaultCloudImages(split="train", transform=display_transform)


In [ ]:
def plot_image(image):
    plt.imshow(image.permute(1, 2, 0))
    plt.axis("off")

In [ ]:
sample_clouds = {}
for img, y in clouds_to_display:
    if y not in sample_clouds:
        sample_clouds[y] = img
    if len(sample_clouds) == len(class_names):
        break

sample_clouds = sorted(sample_clouds.items())[:11]

plt.figure(figsize=(10, 6))
for class_id, image in sample_clouds:
    if class_id == 11: break
    plt.subplot(3, 4, class_id + 1)
    plot_image(image)
    plt.title(f"{class_id}: {class_names[class_id]}", fontsize=11)

plt.show()

In [ ]:
import os
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np

if 'train_set' not in globals():
    raise NameError("train_set is not defined. Since the runtime was restarted, please run all preceding cells from the top of the notebook first!")

train_labels_list = [train_set.encoded_labels[i] for i in range(len(train_set))]
class_counts = np.bincount(train_labels_list)
class_weights = 1.0 / class_counts
sample_weights = torch.tensor([class_weights[l] for l in train_labels_list], dtype=torch.float)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

num_workers = 4
train_loader = DataLoader(train_set, batch_size=128, sampler=sampler, num_workers=num_workers, pin_memory=True, persistent_workers=True)
valid_loader = DataLoader(valid_set, batch_size=128, num_workers=num_workers, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_set,  batch_size=128, num_workers=num_workers, pin_memory=True, persistent_workers=True)

In [ ]:
[name for name, child in model.named_children()]

In [ ]:
model.heads

In [ ]:
model.encoder

In [ ]:
model.conv_proj

In [ ]:
n_classes = 11
model.heads[0] = nn.Linear(768, n_classes).to(device)


In [ ]:
for param in model.parameters():
    param.requires_grad = False

for param in model.heads.parameters():
    param.requires_grad = True

In [ ]:
!pip install torchmetrics

In [ ]:
import torchmetrics
from tqdm.auto import tqdm

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=n_classes).to(device)

In [ ]:
def train_phase(model, optimizer, loss_fn, metric, train_loader, valid_loader,
                n_epochs, patience, checkpoint_path, phase_name):
    print(f'\n{"="*60}')
    print(f'  {phase_name}')
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Trainable params: {trainable:,}  |  Max epochs: {n_epochs}  |  Patience: {patience}')
    print(f'{"="*60}')

    history = {'train_losses': [], 'train_metrics': [], 'valid_metrics': []}
    best_val = 0.0
    epochs_no_improve = 0

    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{n_epochs}'):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                y_pred = model(X_batch)
                loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)

        train_loss = total_loss / len(train_loader)
        train_acc  = metric.compute().item()
        val_acc    = evaluate_tm(model, valid_loader, metric).item()
        history['train_losses'].append(train_loss)
        history['train_metrics'].append(train_acc)
        history['valid_metrics'].append(val_acc)

        improved = val_acc > best_val
        print(f'Epoch {epoch+1}/{n_epochs}, '
              f'train loss: {train_loss:.4f}, '
              f'train metric: {train_acc:.4f}, '
              f'valid metric: {val_acc:.4f}'
              + (' *' if improved else ''))

        if improved:
            best_val = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1} (best val: {best_val:.4f})')
                break

    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    print(f'Best val acc: {best_val:.4f}')
    return history

In [ ]:
# Phase 1 — heads only
# Up to 100 epochs, early stopping patience 20.
optimizer_p1 = torch.optim.AdamW(model.heads.parameters(), lr=1e-3)

history_p1 = train_phase(
    model, optimizer_p1, xentropy, accuracy,
    train_loader, valid_loader,
    n_epochs=100, patience=20,
    checkpoint_path='best_model_phase1.pt',
    phase_name='Phase 1 — heads only  (lr=1e-3)',
)


In [ ]:
# Phase 2 — unfreeze last encoder block (layer 11)
# Up to 50 epochs, early stopping patience 15.
for param in model.encoder.layers[-1].parameters():
    param.requires_grad = True

optimizer_p2 = torch.optim.AdamW([
    {'params': model.heads.parameters(),              'lr': 1e-4},
    {'params': model.encoder.layers[-1].parameters(), 'lr': 1e-5},
])

history_p2 = train_phase(
    model, optimizer_p2, xentropy, accuracy,
    train_loader, valid_loader,
    n_epochs=50, patience=15,
    checkpoint_path='best_model_phase2.pt',
    phase_name='Phase 2 — heads + encoder.layers[-1]  (lr 1e-4 / 1e-5)',
)


In [ ]:
# Phase 3 — unfreeze last 4 encoder blocks (layers 8–11)
# Up to 30 epochs, early stopping patience 10.
for layer in model.encoder.layers[-4:-1]:  # layers 8, 9, 10
    for param in layer.parameters():
        param.requires_grad = True

optimizer_p3 = torch.optim.AdamW([
    {'params': model.heads.parameters(),                                                  'lr': 1e-4},
    {'params': model.encoder.layers[-1].parameters(),                                     'lr': 1e-5},
    {'params': [p for l in model.encoder.layers[-4:-1] for p in l.parameters()],         'lr': 5e-6},
])

history_p3 = train_phase(
    model, optimizer_p3, xentropy, accuracy,
    train_loader, valid_loader,
    n_epochs=30, patience=10,
    checkpoint_path='best_model_phase3.pt',
    phase_name='Phase 3 — heads + encoder.layers[-4:]  (lr 1e-4 / 1e-5 / 5e-6)',
)


In [ ]:
test_acc = evaluate_tm(model, test_loader, accuracy)
print(f'Test accuracy: {test_acc:.4f}')


In [ ]:
import matplotlib.pyplot as plt

phase_histories = list(filter(None, [
    (history_p1, 'Phase 1'),
    (history_p2, 'Phase 2'),
    (history_p3, 'Phase 3'),
]))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = [('tab:blue', 'tab:cyan'), ('tab:orange', 'tab:red'), ('tab:green', 'tab:olive')]
offset = 0
for (h, label), (c_train, c_val) in zip(phase_histories, colors):
    epochs = range(offset + 1, offset + len(h['train_metrics']) + 1)
    axes[0].plot(epochs, h['train_losses'], color=c_train, label=f'{label} train loss')
    axes[1].plot(epochs, h['train_metrics'], color=c_train, label=f'{label} train acc')
    axes[1].plot(epochs, h['valid_metrics'], color=c_val,   label=f'{label} val acc', linestyle='--')
    offset += len(h['train_metrics'])

axes[0].set_title('Training loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(fontsize=7)
axes[1].set_title('Accuracy');      axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()
